In [13]:
import json

exiting_word_set = set()
with open('union_words_disyllabic.txt', 'r', encoding='utf-8') as f:
    for line in f.readlines():
        exiting_word_set.add(line.strip())

with open('filtered_prediction_pairs.json', 'r', encoding='utf-8') as f:
    all_words_to_process = json.load(f)

In [14]:
exiting_word_set

{'举报',
 '腊肠',
 '务农',
 '墨鸦',
 '膝盖',
 '佞人',
 '边音',
 '交迭',
 '胃酸',
 '捻度',
 '风耗',
 '伉俪',
 '进犯',
 '响头',
 '定址',
 '乡僻',
 '冷冽',
 '烘手',
 '牙子',
 '鞫问',
 '噗仔',
 '消暑',
 '孤鬼',
 '语声',
 '尿检',
 '谶记',
 '淫逸',
 '短褂',
 '黄鱼',
 '施用',
 '奶名',
 '笔势',
 '谗佞',
 '卡费',
 '溶蚀',
 '极刑',
 '一同',
 '破费',
 '爵士',
 '戆大',
 '把牢',
 '慧心',
 '茬儿',
 '品月',
 '下装',
 '矿石',
 '晦迹',
 '夜跑',
 '同房',
 '篇什',
 '顾问',
 '付酬',
 '误区',
 '式样',
 '整整',
 '凝点',
 '07',
 '凄楚',
 '权臣',
 '吊针',
 '学龄',
 '苍术',
 '排华',
 '落入',
 '特此',
 '道情',
 '全等',
 '矾土',
 '米粒',
 '起场',
 '乳牙',
 '内臣',
 '水线',
 '付款',
 '一片',
 '官人',
 '企及',
 '鹿砦',
 '整人',
 '油亮',
 '跌入',
 '丹佛',
 '美气',
 '间架',
 '高栏',
 '小朱',
 '呈请',
 '今生',
 '混过',
 '牧犬',
 '惬怀',
 '装死',
 '大灶',
 '彩报',
 '新衣',
 '谨慎',
 '侈糜',
 '位置',
 '勾除',
 '三弦',
 '防漏',
 '合办',
 '攘除',
 '台面',
 '制艺',
 '习用',
 '染色',
 '铁锤',
 '憩歇',
 '田坛',
 '醇浓',
 '往前',
 '莲藕',
 '死人',
 '白页',
 '官商',
 '活质',
 '正是',
 '奴性',
 '研讨',
 '同喜',
 '瞒过',
 '水中',
 '点歌',
 '鼻饲',
 '幻觉',
 '曲牌',
 '二副',
 '稗记',
 '瞠视',
 '公示',
 '沉醉',
 '常礼',
 '瘘管',
 '炫耀',
 '公假',
 '耸肩',
 '子法',
 '赶工',
 '幅面',
 '野蛮',
 '两百',
 '老面',

In [15]:
print(len(all_words_to_process))

165102


In [16]:
#步骤1，过滤掉与已有词汇在词形上一致的候选词
words_process_1 = []
for item in all_words_to_process:
    char1 = item['head_id'][0]
    char2 = item['tail_id'][0]
    if char1+char2 not in exiting_word_set:
        words_process_1.append(item)
print(len(words_process_1))

99340


In [17]:
#步骤2，过滤掉“后缀”在前语素，或“前缀”在后语素的词；过滤掉语素词性为缀词但构词结构非“前缀”“后缀”的词
words_process_2 = []
for item in words_process_1:
    char1 = item['head_id'][0]
    char2 = item['tail_id'][0]
    char1_pos = item['head_pos']
    char2_pos = item['tail_pos']
    char1_sense = item['head_sense']
    char2_sense = item['tail_sense']
    if "后缀" in char1_sense and char1_pos == "缀词":
        continue
    if "前缀" in char2_sense and char2_pos == "缀词":
        continue
    if (char1_pos == "缀词" or char2_pos == "缀词") and item['relation'] not in ["前缀", "后缀"]:
        continue
    
    words_process_2.append(item)

In [18]:
print(len(words_process_2))

98616


In [19]:
#step3 找到已知词的所有(前pos, 后pos, 构词结构)三元组，排除掉所有不在这些三元组中的候选词


morpheme_dict = {}
with open('../data/1_语素.txt','r',encoding='utf-8') as f:
    lines = f.readlines()
    for line in lines:
        line=line.split()
        morpheme_dict[line[0]]=line
existing_word_tuples = set()
with open('../data/3_语法构词结构.txt','r', encoding='utf-8') as f:
    for line in f.readlines():
        line = line.strip().split()
        pos1 = morpheme_dict[line[3][2:]][4][:2]
        pos2 = morpheme_dict[line[4][2:]][4][:2]
        formation = line[2]
        existing_word_tuples.add((pos1, pos2, formation))

words_process_3 = []
for item in words_process_2:
    char1_pos = item['head_pos']
    char2_pos = item['tail_pos']
    formation = item['relation']
    if (char1_pos, char2_pos, formation) in existing_word_tuples:
        words_process_3.append(item)
print(len(words_process_3))

73381


In [20]:
words_process_3[:20]

[{'head_id': '心02795_1_05_02',
  'head_pos': '名词',
  'head_sense': '通常也指思想的器官和思想、感情等。',
  'relation': '主谓',
  'tail_id': '挂03306_1_09_06',
  'tail_pos': '动词',
  'tail_sense': '牵挂。',
  'confidence': 0.78759765625},
 {'head_id': '心02795_1_05_02',
  'head_pos': '名词',
  'head_sense': '通常也指思想的器官和思想、感情等。',
  'relation': '主谓',
  'tail_id': '顾09847_1_06_05',
  'tail_pos': '动词',
  'tail_sense': '珍惜；顾念。',
  'confidence': 0.9931640625},
 {'head_id': '心02795_1_05_02',
  'head_pos': '名词',
  'head_sense': '通常也指思想的器官和思想、感情等。',
  'relation': '主谓',
  'tail_id': '感03010_1_06_02',
  'tail_pos': '动词',
  'tail_sense': '被某事或某人所触动，引起情感上的共鸣或激动。',
  'confidence': 0.9951171875},
 {'head_id': '心02795_1_05_02',
  'head_pos': '名词',
  'head_sense': '通常也指思想的器官和思想、感情等。',
  'relation': '主谓',
  'tail_id': '戒03088_1_07_01',
  'tail_pos': '动词',
  'tail_sense': '防备；警惕。',
  'confidence': 0.7109375},
 {'head_id': '心02795_1_05_02',
  'head_pos': '名词',
  'head_sense': '通常也指思想的器官和思想、感情等。',
  'relation': '主谓',
  'tail_id': '恋02

In [21]:
with open('filtered_prediction_pairs_step3.json', 'w', encoding='utf-8') as f:
    json.dump(words_process_3, f, ensure_ascii=False, indent=4)

In [31]:
words_process_3_word_dict = {}
for item in words_process_3:
    word = item['head_id'] + '+' +item['tail_id']

    if word not in words_process_3_word_dict:
        words_process_3_word_dict[word] = []
    words_process_3_word_dict[word].append(item)

In [32]:
print(len(words_process_3_word_dict))

46998


In [33]:
with open('filtered_prediction_pairs_step3_dict.json', 'w', encoding='utf-8') as f:
    json.dump(words_process_3_word_dict, f, ensure_ascii=False, indent=4)

In [34]:
words_process_4 = []
for word in words_process_3_word_dict:
    candidates = words_process_3_word_dict[word]
    if len(candidates) == 1:
        words_process_4.append(candidates[0])
    else:
        # Find the candidate with maximum confidence
        best_candidate = max(candidates, key=lambda x: x['confidence'])
        words_process_4.append(best_candidate)

In [ ]:
with open('filtered_prediction_pairs_step4.json', 'w', encoding='utf-8') as f:
    json.dump(words_process_4, f, ensure_ascii=False, indent=4)